In [0]:
%run "./00.Helper Function"

In [0]:
import requests
import json
from datetime import datetime
from datetime import date

In [0]:
# Creating the scope and api-key secert for youtube api run it one time only 
# create_scope("youtube", "DATABRICKS")
# create_secret("youtube", "api_key", "value_of_api_key")


In [0]:
def get_playlist_id(channel_handle:str):
    api_key=dbutils.secrets.get(scope="youtube", key="api_key")
    try:
        url=f"https://youtube.googleapis.com/youtube/v3/channels?part=contentDetails&forHandle={channel_handle}&key={api_key}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        channel_items = data["items"][0]
        playlist_id = channel_items["contentDetails"]["relatedPlaylists"]["uploads"]
        return playlist_id
    except requests.exceptions.RequestException as e:
        raise Exception(f"Error fetching playlist ID: {e}")

In [0]:
def get_video_ids(playlistId:str):
    api_key=dbutils.secrets.get(scope="youtube", key="api_key")
    video_ids = []
    pageToken = None
    maxResults = 50
    try:
        url=f"https://youtube.googleapis.com/youtube/v3/playlistItems?part=contentDetails&maxResults={maxResults}&key={api_key}&playlistId={playlistId}"
        while True:
            if pageToken:
                url += f"&pageToken={pageToken}"
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            for item in data.get("items", []):
                video_id = item["contentDetails"]["videoId"]
                video_ids.append(video_id)
            pageToken = data.get("nextPageToken")
            if not pageToken:
                break
        return video_ids
    except requests.exceptions.RequestException as e:
        raise Exception(f"Error fetching video IDs: {e}")

In [0]:
def extract_video_data(video_ids):
    api_key = dbutils.secrets.get(scope="youtube", key="api_key")
    extracted_data = []
    maxResults = 50

    def batch_list(video_id_lst, batch_size):
        for i in range(0, len(video_id_lst), batch_size):
            yield video_id_lst[i : i + batch_size]

    try:
        for batch in batch_list(video_ids, maxResults):
            video_ids_str = ",".join(batch)
            url = f"https://youtube.googleapis.com/youtube/v3/videos?part=contentDetails,snippet,statistics&id={video_ids_str}&key={api_key}"
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            for item in data.get("items", []):
                video_id = item["id"]
                snippet = item.get("snippet", {})
                contentDetails = item.get("contentDetails", {})
                statistics = item.get("statistics", {})

                video_data = {
                    "video_id": video_id,
                    "title": snippet.get("title"),
                    "publishedAt": snippet.get("publishedAt"),
                    "duration": contentDetails.get("duration"),
                    "viewCount": statistics.get("viewCount"),
                    "likeCount": statistics.get("likeCount"),
                    "favoriteCount": statistics.get("favoriteCount"),
                    "commentCount": statistics.get("commentCount"),
                }
                extracted_data.append(video_data)
        return extracted_data
    except requests.exceptions.RequestException as e:
        raise Exception(f"Error fetching video data: {e}")


In [0]:
def save_to_json(extracted_data):
    volume_path = "/Volumes/youtube/landing/data/files"
    file_path = f"{volume_path}/yt_data_{date.today()}_{datetime.now().timestamp()}.json"
    with open(file_path, "w", encoding="utf-8") as json_outfile:
        json.dump(extracted_data, json_outfile, indent=4, ensure_ascii=False)
    print(f" file: {file_path} saved successfully")

In [0]:
playlistId=get_playlist_id(channel_handle="MrBeast")
video_ids = get_video_ids(playlistId)
extracted_data = extract_video_data(video_ids)
save_to_json(extracted_data)
